In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Raw dataset
file_path = Path("../01_Raw_Data/marketing_data.xlsx")

df_raw = pd.read_excel(file_path)

# Orijinal datanın surətini yaradırıq
df = df_raw.copy()

print("Raw Data Shape:", df_raw.shape)
print("Working Data Shape:", df.shape)

Raw Data Shape: (2240, 28)
Working Data Shape: (2240, 28)


In [2]:
# Sütun adlarını standartlaşdırırıq
df.columns = df.columns.str.strip()

# Yoxlama
print(df.columns.tolist())

['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Response', 'Complain', 'Country']


In [3]:
quality_summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Null_Count": df.isnull().sum().values,
    "Null_Percentage": (
        df.isnull().sum().values / len(df) * 100
    ).round(2),
    "Unique_Values": df.nunique().values
})

display(quality_summary)

,Column,Data_Type,Null_Count,Null_Percentage,Unique_Values
0,ID,int64,0,0.00,2240
1,Year_Birth,int64,0,0.00,59
2,Education,str,0,0.00,5
3,Marital_Status,str,0,0.00,8
4,Income,float64,24,1.07,1974
5,Kidhome,int64,0,0.00,3
6,Teenhome,int64,0,0.00,3
7,Dt_Customer,datetime64[us],0,0.00,663
8,Recency,int64,0,0.00,100
9,MntWines,int64,0,0.00,776


In [5]:
cleaning_log = []

cleaning_log.append({
    "Step": 1,
    "Column": "Income",
    "Issue": "Column name contains spaces",
    "Action": "Remove leading and trailing spaces",
    "Method": "str.strip()",
    "Status": "Completed"
})

cleaning_log = pd.DataFrame(cleaning_log)

display(cleaning_log)

,Step,Column,Issue,Action,Method,Status
0,1,Income,Column name contains spaces,Remove leading and trailing spaces,str.strip(),Completed


In [6]:
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percent": (
        df.isna().sum().values / len(df) * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
]

display(missing_summary)

,Column,Missing_Count,Missing_Percent
4,Income,24,1.07


In [7]:
missing_income = df[df["Income"].isna()]

display(
    missing_income[
        [
            "ID",
            "Year_Birth",
            "Education",
            "Marital_Status",
            "Income",
            "Country"
        ]
    ]
)

,ID,Year_Birth,Education,Marital_Status,Income,Country
134,8996,1957,PhD,Married,NaN,Germany
262,1994,1983,Graduation,Married,NaN,USA
394,3769,1972,PhD,Together,NaN,Australia
449,5255,1986,Graduation,Single,NaN,Australia
525,8268,1961,PhD,Married,NaN,Canada
590,10629,1973,2n Cycle,Married,NaN,Germany
899,10475,1970,Master,Together,NaN,USA
997,9235,1957,Graduation,Single,NaN,Germany
1096,4345,1964,2n Cycle,Single,NaN,Australia
1185,7187,1969,Master,Together,NaN,Australia


In [8]:
missing_by_education = (
    df.groupby("Education", dropna=False)["Income"]
    .agg(
        Total_Customers="size",
        Missing_Income=lambda x: x.isna().sum()
    )
)

missing_by_education["Missing_Rate"] = (
    missing_by_education["Missing_Income"]
    / missing_by_education["Total_Customers"]
    * 100
).round(2)

display(missing_by_education)

,Total_Customers,Missing_Income,Missing_Rate
Education,,,
2n Cycle,203,3,1.48
Basic,54,0,0.00
Graduation,1127,11,0.98
Master,370,5,1.35
PhD,486,5,1.03


In [9]:
print("Mean Income:", round(df["Income"].mean(), 2))
print("Median Income:", round(df["Income"].median(), 2))

print("Minimum Income:", df["Income"].min())
print("Maximum Income:", df["Income"].max())

Mean Income: 52247.25
Median Income: 51381.5
Minimum Income: 1730.0
Maximum Income: 666666.0


In [10]:
df_drop = df.dropna(subset=["Income"])

print("Before:", len(df))
print("After:", len(df_drop))

Before: 2240
After: 2216


In [11]:
df_mean = df.copy()

mean_income = df_mean["Income"].mean()

df_mean["Income"] = (
    df_mean["Income"].fillna(mean_income)
)

In [12]:
df_median = df.copy()

median_income = df_median["Income"].median()

df_median["Income"] = (
    df_median["Income"].fillna(median_income)
)

In [13]:
comparison = pd.DataFrame({
    "Method": [
        "Original",
        "Drop NULL",
        "Mean Imputation",
        "Median Imputation"
    ],
    "Rows": [
        len(df),
        len(df_drop),
        len(df_mean),
        len(df_median)
    ],
    "Missing": [
        df["Income"].isna().sum(),
        df_drop["Income"].isna().sum(),
        df_mean["Income"].isna().sum(),
        df_median["Income"].isna().sum()
    ],
    "Mean_Income": [
        df["Income"].mean(),
        df_drop["Income"].mean(),
        df_mean["Income"].mean(),
        df_median["Income"].mean()
    ],
    "Median_Income": [
        df["Income"].median(),
        df_drop["Income"].median(),
        df_mean["Income"].median(),
        df_median["Income"].median()
    ]
})

comparison["Mean_Income"] = (
    comparison["Mean_Income"].round(2)
)

comparison["Median_Income"] = (
    comparison["Median_Income"].round(2)
)

display(comparison)

,Method,Rows,Missing,Mean_Income,Median_Income
0,Original,2240,24,52247.25,51381.5
1,Drop NULL,2216,0,52247.25,51381.5
2,Mean Imputation,2240,0,52247.25,51741.5
3,Median Imputation,2240,0,52237.98,51381.5


In [14]:
# Əvvəlcə NULL statusunu qeyd edirik
df["Income_Was_Missing"] = (
    df["Income"].isna().astype(int)
)

# Median gəliri hesablayırıq
median_income = df["Income"].median()

# Yeni analitik sütun
df["Income_Imputed"] = (
    df["Income"].fillna(median_income)
)

display(
    df[
        [
            "ID",
            "Income",
            "Income_Was_Missing",
            "Income_Imputed"
        ]
    ].head(10)
)

,ID,Income,Income_Was_Missing,Income_Imputed
0,1826,84835.0,0,84835.0
1,1,57091.0,0,57091.0
2,10476,67267.0,0,67267.0
3,1386,32474.0,0,32474.0
4,5371,21474.0,0,21474.0
5,7348,71691.0,0,71691.0
6,4073,63564.0,0,63564.0
7,1991,44931.0,0,44931.0
8,4047,65324.0,0,65324.0
9,9477,65324.0,0,65324.0


In [15]:
print(
    "Original Income NULL:",
    df["Income"].isna().sum()
)

print(
    "Imputed Income NULL:",
    df["Income_Imputed"].isna().sum()
)

print(
    "Missing Flag Count:",
    df["Income_Was_Missing"].sum()
)

print(
    "Total Rows:",
    len(df)
)

Original Income NULL: 24
Imputed Income NULL: 0
Missing Flag Count: 24
Total Rows: 2240


In [16]:
new_log = pd.DataFrame([{
    "Step": 2,
    "Column": "Income",
    "Issue": "Missing Values",
    "Action": "Create imputed analytical column",
    "Method": "Median Imputation",
    "Status": "Completed"
}])

cleaning_log = pd.concat(
    [cleaning_log, new_log],
    ignore_index=True
)

display(cleaning_log)

,Step,Column,Issue,Action,Method,Status
0,1,Income,Column name contains spaces,Remove leading and trailing spaces,str.strip(),Completed
1,2,Income,Missing Values,Create imputed analytical column,Median Imputation,Completed


In [17]:
# Tamamilə eyni sətirlərin sayı
exact_duplicates = df.duplicated().sum()

print("Exact Duplicate Rows:", exact_duplicates)

Exact Duplicate Rows: 0


In [18]:
duplicate_rows = df[
    df.duplicated(keep=False)
]

display(duplicate_rows)

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country,Income_Was_Missing,Income_Imputed


In [19]:
duplicate_ids = df["ID"].duplicated().sum()

print("Duplicate Customer IDs:", duplicate_ids)

Duplicate Customer IDs: 0


In [20]:
rows_before = len(df)

df = df.drop_duplicates().copy()

rows_after = len(df)

print("Rows Before:", rows_before)
print("Rows After:", rows_after)
print("Removed Rows:", rows_before - rows_after)

Rows Before: 2240
Rows After: 2240
Removed Rows: 0


In [21]:
print(
    "Remaining Exact Duplicates:",
    df.duplicated().sum()
)

print(
    "Remaining Duplicate IDs:",
    df["ID"].duplicated().sum()
)

Remaining Exact Duplicates: 0
Remaining Duplicate IDs: 0


In [22]:
invalid_recency = df[
    df["Recency"] < 0
]

print(
    "Invalid Recency Records:",
    len(invalid_recency)
)

display(invalid_recency[
    ["ID", "Recency"]
])

Invalid Recency Records: 0


,ID,Recency


In [23]:
invalid_household = df[
    (df["Kidhome"] < 0) |
    (df["Teenhome"] < 0)
]

print(
    "Invalid Household Records:",
    len(invalid_household)
)

display(
    invalid_household[
        ["ID", "Kidhome", "Teenhome"]
    ]
)

Invalid Household Records: 0


,ID,Kidhome,Teenhome


In [24]:
campaign_columns = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
    "Response",
    "Complain"
]

campaign_validation = {}

for col in campaign_columns:

    invalid_count = (
        ~df[col].isin([0, 1])
    ).sum()

    campaign_validation[col] = invalid_count

display(
    pd.DataFrame(
        campaign_validation.items(),
        columns=["Column", "Invalid_Count"]
    )
)

,Column,Invalid_Count
0,AcceptedCmp1,0
1,AcceptedCmp2,0
2,AcceptedCmp3,0
3,AcceptedCmp4,0
4,AcceptedCmp5,0
5,Response,0
6,Complain,0


In [25]:
print(df["Dt_Customer"].dtype)

print(
    "Minimum Date:",
    df["Dt_Customer"].min()
)

print(
    "Maximum Date:",
    df["Dt_Customer"].max()
)

datetime64[us]
Minimum Date: 2012-07-30 00:00:00
Maximum Date: 2014-06-29 00:00:00


In [26]:
registration_dates = pd.to_datetime(
    df["Dt_Customer"],
    errors="coerce"
)

invalid_dates = df[
    registration_dates > pd.Timestamp.today().normalize()
]

print(
    "Future Registration Dates:",
    len(invalid_dates)
)

Future Registration Dates: 0


In [27]:
consistency_summary = pd.DataFrame({
    "Check": [
        "Exact Duplicate",
        "Duplicate Customer ID",
        "Negative Recency",
        "Invalid Household",
        "Invalid Campaign Values",
        "Future Registration Date"
    ],
    "Issue_Count": [
        int(df.duplicated().sum()),
        int(df["ID"].duplicated().sum()),
        int((df["Recency"] < 0).sum()),
        int(
            (
                (df["Kidhome"] < 0) |
                (df["Teenhome"] < 0)
            ).sum()
        ),
        int(
            sum(campaign_validation.values())
        ),
        int(len(invalid_dates))
    ]
})

consistency_summary["Status"] = np.where(
    consistency_summary["Issue_Count"] == 0,
    "PASS",
    "FAILED"
)

display(consistency_summary)

,Check,Issue_Count,Status
0,Exact Duplicate,0,PASS
1,Duplicate Customer ID,0,PASS
2,Negative Recency,0,PASS
3,Invalid Household,0,PASS
4,Invalid Campaign Values,0,PASS
5,Future Registration Date,0,PASS


In [28]:
new_logs = pd.DataFrame([
    {
        "Step": 3,
        "Column": "All Columns",
        "Issue": "Exact Duplicate Records",
        "Action": "Check and remove exact duplicates",
        "Method": "duplicated() / drop_duplicates()",
        "Status": "Completed"
    },
    {
        "Step": 4,
        "Column": "ID",
        "Issue": "Duplicate Customer IDs",
        "Action": "Validate unique customer identifiers",
        "Method": "duplicated(subset=['ID'])",
        "Status": "Completed"
    },
    {
        "Step": 5,
        "Column": "Multiple Columns",
        "Issue": "Data Consistency",
        "Action": "Validate business rules",
        "Method": "Conditional Validation",
        "Status": "Completed"
    }
])

cleaning_log = pd.concat(
    [cleaning_log, new_logs],
    ignore_index=True
)

display(cleaning_log)

,Step,Column,Issue,Action,Method,Status
0,1,Income,Column name contains spaces,Remove leading and trailing spaces,str.strip(),Completed
1,2,Income,Missing Values,Create imputed analytical column,Median Imputation,Completed
2,3,All Columns,Exact Duplicate Records,Check and remove exact duplicates,duplicated() / drop_duplicates(),Completed
3,4,ID,Duplicate Customer IDs,Validate unique customer identifiers,duplicated(subset=['ID']),Completed
4,5,Multiple Columns,Data Consistency,Validate business rules,Conditional Validation,Completed
